<a href="https://colab.research.google.com/github/vsujancs24-max/AI-LAB/blob/main/1WN24CS312_Week9_FOL_Resolution%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from itertools import combinations


def negate_literal(literal):
    return literal[1:] if literal.startswith("~") else f"~{literal}"


def format_clause(clause):
    if not clause:
        return "□ (Empty Clause)"
    return "(" + " ∨ ".join(sorted(clause)) + ")"


def resolve(ci, cj):
    resolvents = []
    for literal in ci:
        neg = negate_literal(literal)
        if neg in cj:
            new_clause = (ci - {literal}) | (cj - {neg})
            resolvents.append(frozenset(new_clause))
    return resolvents


def print_resolution_tree(clause, parent_map, indent="", is_last=True):
    branch = "└── " if is_last else "├── "
    source = (
        "" if clause in parent_map else "  [Given Axiom / Negated Query]"
    )
    print(f"{indent}{branch}{format_clause(clause)}{source}")

    if clause in parent_map:
        p1, p2 = parent_map[clause]
        new_indent = indent + ("    " if is_last else "│   ")
        print_resolution_tree(p1, parent_map, new_indent, is_last=False)
        print_resolution_tree(p2, parent_map, new_indent, is_last=True)


def resolution_prove(kb, query):
    negated_query = frozenset({negate_literal(query)})
    clauses = set(kb) | {negated_query}
    parent_map = {}

    while True:
        new_clauses = set()

        for ci, cj in combinations(clauses, 2):
            resolvents = resolve(ci, cj)
            for res in resolvents:
                if res not in clauses and res not in parent_map:
                    parent_map[res] = (ci, cj)

                if len(res) == 0:
                    print("Query Proven! Resolution Tree:\n")
                    print_resolution_tree(res, parent_map)
                    return True

                new_clauses.add(res)

        if new_clauses.issubset(clauses):
            print("Query cannot be proven from the Knowledge Base.")
            return False

        clauses.update(new_clauses)


if __name__ == "__main__":
    knowledge_base = [
        frozenset({"~R", "W"}),
        frozenset({"~W", "S"}),
        frozenset({"~S", "C"}),
        frozenset({"R"}),
    ]

    query_to_prove = "C"

    resolution_prove(knowledge_base, query_to_prove)

Query Proven! Resolution Tree:

└── □ (Empty Clause)
    ├── (R)  [Given Axiom / Negated Query]
    └── (~R)
        ├── (~S)
        │   ├── (C ∨ ~S)  [Given Axiom / Negated Query]
        │   └── (~C)  [Given Axiom / Negated Query]
        └── (S ∨ ~R)
            ├── (S ∨ ~W)  [Given Axiom / Negated Query]
            └── (W ∨ ~R)  [Given Axiom / Negated Query]
